In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv('laptop_price.csv', encoding='latin1')

print(df[['Company', 'TypeName', 'Ram', 'Weight', 'Price_euros']].head() if 'Price_euros' in df.columns else df[['Company', 'TypeName', 'Ram', 'Weight', 'Price']].head(), "\n")

df['Ram'] = df['Ram'].str.replace('GB', '').astype(int)
df['Weight'] = df['Weight'].str.replace('kg', '').astype(float)

df['CPU_Speed_GHz'] = df['Cpu'].str.extract(r'(\d+\.\d+|\d+)GHz').astype(float)
df['CPU_Speed_GHz'] = df['CPU_Speed_GHz'].fillna(df['CPU_Speed_GHz'].median())

price_col = 'Price_euros' if 'Price_euros' in df.columns else 'Price'

df['Price_INR'] = df[price_col] * 90.0


X = df[['Ram', 'Weight', 'CPU_Speed_GHz']]
y = df['Price_INR']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f" Mean Absolute Error (MAE): ₹{mae:.2f}")
print(f" R-squared Score: {r2 * 100:.2f}%")

print(f"Base intercept price (c): ₹{model.intercept_:.2f}")
for feature, coef in zip(X.columns, model.coef_):
    print(f"Coefficient for [{feature}] (m): ₹{coef:.2f}")


comparison_df = pd.DataFrame({
    'Actual Price (INR)': y_test.round(2),
    'Predicted Price (INR)': y_pred.round(2)
})

comparison_df['Difference (Error)'] = (comparison_df['Actual Price (INR)'] - comparison_df['Predicted Price (INR)']).abs().round(2)

print("--- Actual vs. Predicted Price Comparison (First 10 Test Samples) ---")
print(comparison_df.head(10).to_string())



  Company   TypeName   Ram  Weight  Price_euros
0   Apple  Ultrabook   8GB  1.37kg      1339.69
1   Apple  Ultrabook   8GB  1.34kg       898.94
2      HP   Notebook   8GB  1.86kg       575.00
3   Apple  Ultrabook  16GB  1.83kg      2537.45
4   Apple  Ultrabook   8GB  1.37kg      1803.60 

 Mean Absolute Error (MAE): ₹28568.58
 R-squared Score: 59.46%
Base intercept price (c): ₹-6500.47
Coefficient for [Ram] (m): ₹8754.04
Coefficient for [Weight] (m): ₹-12161.64
Coefficient for [CPU_Speed_GHz] (m): ₹25959.43
--- Actual vs. Predicted Price Comparison (First 10 Test Samples) ---
      Actual Price (INR)  Predicted Price (INR)  Difference (Error)
479             150480.0              115660.71            34819.29
1022            103410.0              114931.02            11521.02
298              44910.0               79138.46            34228.46
1265             80910.0               99406.11            18496.11
774             111960.0               72898.71            39061.29
115      